<a href="https://colab.research.google.com/github/FarhanKhan1/pytorch_implementations/blob/main/gpt_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a GPT

In [1]:
#My block
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-23 12:49:56--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2026-07-23 12:49:56 (7.18 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
#My block
with open("input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
#My block
print(f"Characters in my Dataset: {len(text)}")

Characters in my Dataset: 1115394


In [4]:
#My block
print("First 500 characters in My dataset: ", text[:500])

First 500 characters in My dataset:  First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [5]:
#My block
#let's say we add all unique characters that can build almost any word in this dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [6]:
#My block
# Let us create the integer mapping of our characters, i.e: {char1: int, char2:int, ...}
stoi = {character:index for index, character in enumerate(chars)} #str to output int
itos = {index: character for index, character in enumerate(chars)} #int to str

encode = lambda any_string: [stoi[ch] for ch in any_string] #it takes any_string and generates input_ids
decode = lambda input_ids: "".join([itos[input_id] for input_id in input_ids]) #it takes input_ids and generates string

In [7]:
#My block
input_ids = encode("Hello world!") #generate input IDs
print(f"Input IDs: {input_ids}")
print(f"original string: {decode(input_ids)}") #regenerate the original string

Input IDs: [20, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
original string: Hello world!


In [8]:
#My block
#As we will be working with Tensors, let's generate a Tensor of Input IDs from our text corpus
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [9]:
#My block
import numpy as np
print(data.shape, data.dtype)
print((np.array(data)[-1:-5:-1]))

torch.Size([1115394]) torch.int64
[ 0  8 45 52]


In [10]:
#My block
#let's build our training and validation data, we will split 90% to train and remaining 10% for validation
split_size = int(0.9 * len(data))
train_data = data[:split_size]
val_data = data[split_size:]

In [11]:
#My block
print(train_data.shape)
print(val_data.shape)

torch.Size([1003854])
torch.Size([111540])


In [12]:
#My block
#Let us create a dataset in generative models format, where all previous words generates the target word
block_size = 8
train_data[:block_size+1]

x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
  context = x[:i+1]
  target = y[i]
  print(f"when input is {context} the target: {target}")
  print(f"when input is {decode(context.tolist())} the target: {decode([target.item()])}")

when input is tensor([18]) the target: 47
when input is F the target: i
when input is tensor([18, 47]) the target: 56
when input is Fi the target: r
when input is tensor([18, 47, 56]) the target: 57
when input is Fir the target: s
when input is tensor([18, 47, 56, 57]) the target: 58
when input is Firs the target: t
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is First the target:  
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is First  the target: C
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is First C the target: i
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58
when input is First Ci the target: t


In [13]:
#Myblock
torch.manual_seed(1337)
batch_size = 6 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
  data = train_data if split == 'train' else val_data
  random_starting_points = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
  x = torch.stack([data[start_point:start_point+block_size] for start_point in random_starting_points])
  y = torch.stack([data[start_point+1:start_point+block_size+1] for start_point in random_starting_points])
  return x, y

xb, yb = get_batch('train')

for batch_num in range(batch_size):
  for block_num in range(block_size):
    context = xb[batch_num,:block_num+1]
    target = yb[batch_num,block_num]
    print(f"when input is {context} the target: {target}")
    # print(f"when input is {decode(context.tolist())} the target: {decode([target.item()])}")

when input is tensor([24]) the target: 43
when input is tensor([24, 43]) the target: 58
when input is tensor([24, 43, 58]) the target: 5
when input is tensor([24, 43, 58,  5]) the target: 57
when input is tensor([24, 43, 58,  5, 57]) the target: 1
when input is tensor([24, 43, 58,  5, 57,  1]) the target: 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]) the target: 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target: 39
when input is tensor([44]) the target: 53
when input is tensor([44, 53]) the target: 56
when input is tensor([44, 53, 56]) the target: 1
when input is tensor([44, 53, 56,  1]) the target: 58
when input is tensor([44, 53, 56,  1, 58]) the target: 46
when input is tensor([44, 53, 56,  1, 58, 46]) the target: 39
when input is tensor([44, 53, 56,  1, 58, 46, 39]) the target: 58
when input is tensor([44, 53, 56,  1, 58, 46, 39, 58]) the target: 1
when input is tensor([52]) the target: 58
when input is tensor([52, 58]) the target: 1
when input is tensor(

In [14]:
#My block
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54],
        [57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1]])


In [15]:
xb.shape

torch.Size([6, 8])

In [16]:
yb

tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39],
        [43, 60, 43, 52,  1, 63, 43, 39],
        [43, 42,  8,  0, 25, 63,  1, 45]])

In [43]:
batch_size = 64
block_size = 256
n_embeddings = 32
head_size = 32
dropout = 0.0
n_layer = 4
n_head = 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 1e-3
max_iters = 5000
eval_interval = 500
eval_iters = 500

In [44]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embeddings, head_size, bias=False)
        self.query = nn.Linear(n_embeddings, head_size, bias=False)
        self.value = nn.Linear(n_embeddings, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embeddings, n_embeddings)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embeddings)
        self.position_embedding_table = nn.Embedding(block_size, n_embeddings)
        self.sa_head = nn.Sequential(*[Block(n_embeddings, n_head) for _ in range(n_layer)])
        self.ffwd = FeedFoward(n_embeddings)
        self.ln_f = nn.LayerNorm(n_embeddings)
        self.lm_head = nn.Linear(n_embeddings, vocab_size)

    def forward(self, idx, targets=None):

        
        token_embeddings = self.token_embedding_table(idx)
        position_embeddings = self.position_embedding_table(torch.arange(start=0, end=idx.shape[1], device=idx.device)) # (T,C)
        x = token_embeddings + position_embeddings # (B,T,C)
        x = self.sa_head(x) # (B,T,C)
        x = self.ffwd(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# m = BigramLanguageModel(vocab_size)
# logits, loss = m(xb, yb)
# print(logits.shape)
# print(loss)

# print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


In [47]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:

model = BigramLanguageModel(vocab_size)
m = model.to(device)
# Total number of parameters
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

"""
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for steps in range(1000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())
"""

0.063265 M parameters
step 0: train loss 4.2535, val loss 4.2568
step 500: train loss 2.6186, val loss 2.6150
step 1000: train loss 2.4780, val loss 2.4681
step 1500: train loss 2.4031, val loss 2.3656
step 2000: train loss 2.3604, val loss 2.3709
step 2500: train loss 2.2938, val loss 2.2938
step 3000: train loss 2.2875, val loss 2.2867
step 3500: train loss 2.2276, val loss 2.2785
step 4000: train loss 2.2168, val loss 2.2271
step 4500: train loss 2.1818, val loss 2.2141
step 5000: train loss 2.1864, val loss 2.2306
step 5500: train loss 2.1709, val loss 2.1896
step 6000: train loss 2.1102, val loss 2.1799
step 6500: train loss 2.1105, val loss 2.1948
step 7000: train loss 2.1074, val loss 2.1597
step 7500: train loss 2.1104, val loss 2.1578
step 8000: train loss 2.1019, val loss 2.1458
step 8500: train loss 2.1016, val loss 2.1307
step 9000: train loss 2.0900, val loss 2.1500
step 9500: train loss 2.0627, val loss 2.1160
step 9999: train loss 2.0679, val loss 2.1429


"\n# create a PyTorch optimizer\noptimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)\n\nbatch_size = 32\nfor steps in range(1000): # increase number of steps for good results...\n\n    # sample a batch of data\n    xb, yb = get_batch('train')\n\n    # evaluate the loss\n    logits, loss = m(xb, yb)\n    optimizer.zero_grad(set_to_none=True)\n    loss.backward()\n    optimizer.step()\n\nprint(loss.item())\n"

In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

# **Let's create the model now using the self attention**

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [ ]:
xbow = torch.zeros((B,T,C))
print(xbow)


In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
      # print(f"x_previous= {x[b,:t+1]}")
      xprev = x[b,:t+1] # (t,C)
      xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
xbow[0]

In [ ]:
x[0]

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow_1 = wei @ x

In [ ]:
trill = torch.tril(torch.ones(T, T))

wei = torch.zeros(T, T)

wei = wei.masked_fill(trill==0, float('-inf'))

softmax = torch.nn.Softmax(dim=1)

softmax(wei)

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

In [ ]:
wei[0]

In [ ]:
v[0,:,:9]

In [ ]:
out[0]